<table>
<tr>                                                                                   
     <th>
         <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'>Spark-DataFrames SQL</div>
     </th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/spark/presentation.png" width="256">
     </th>
 </tr>
</table>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("06-API.DataFrames-SQL")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/10/19 11:56:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 11:56:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/19 11:56:29 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20261019115627-0006/events_1_app-20261019115627-0006.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20261019115627-0006
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Lecture et consolidation de données</div></b>

In [4]:
ROOT = "s3a://lakehouse"
donnees = {}
for nom in ["fact_logistics","fact_order","fact_order_item","fact_payment","fact_review","mart_customer_rfm"]:
    donnees[nom] = spark.read.parquet(f"{ROOT}/silver/ebrasil/{nom}")
    # print(donnees[nom].printSchema())

In [5]:
donnees["fact_logistics"].toPandas().head(5)

26/10/19 11:56:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

,order_id,customer_id,customer_unique_id,representative_seller_id,customer_zip,seller_zip,status,purchase_date_key,approved_date_key,shipped_date_key,...,estimated_delivery_date_weekday,estimated_delivery_date_is_weekend,purchase_time_hour,purchase_time_minute,purchase_time_second,purchase_time_day_part,delivered_time_hour,delivered_time_minute,delivered_time_second,delivered_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,914991f0c02ef0843c0e7010c819d642,0000f6ccb0745a6a4b88665a16c9f078,ef506c96320abeedfb894c34db06f478,66812,3569,delivered,20171012,20171012.0,20171013.0,...,2,False,20,29,41,evening,21.0,23.0,5.0,evening
1,d957021f1127559cd947b62533f484f7,47227568b10f5f58a524a75507e6992c,0004aac84e0df4da2b147fca70cf8255,70a12e78e608ac31179aea7f8422044b,18040,12327,delivered,20171114,20171114.0,20171116.0,...,3,False,19,45,42,evening,23.0,8.0,56.0,evening
2,d0028facea13f508e880202d7097a5a1,d2509c13692836fc0449e88cf9eb4858,00050ab1314c0e55a6ca13cf7181fecf,ea8482cd71df3c1969d7b9473ff13abc,13084,4160,delivered,20180420,20180425.0,20180425.0,...,4,False,12,57,23,afternoon,12.0,8.0,59.0,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,3b37fb626fdf46cd99d37ec62afa88ff,0005e1862207bf6ccc02e4228effd9a0,0cbcee27c791afa0cdcb08587a2013a8,25966,37410,delivered,20170304,20170304.0,20170306.0,...,5,False,23,32,12,evening,8.0,33.0,8.0,morning
4,6681163e3dab91c549952b2845b20281,2f29573c8cac5a7be11c5b649078f944,0006fdc98a402fceb4eb0ee528f6a8d4,d2374cbcbb3ca4ab1086534108cc3ab7,29400,14940,delivered,20170718,20170718.0,20170721.0,...,3,False,9,23,10,morning,18.0,42.0,49.0,evening


In [6]:
donnees["fact_order"].toPandas().head(5)

,order_id,customer_unique_id,status,purchase_date_key,approved_date_key,shipped_date_key,delivered_date_key,estimated_delivery_date_key,purchase_time_key,approved_time_key,...,approved_time_second,approved_time_day_part,shipped_time_hour,shipped_time_minute,shipped_time_second,shipped_time_day_part,delivered_time_hour,delivered_time_minute,delivered_time_second,delivered_time_day_part
0,3e470077b690ea3e3d501cffb5e0c499,0004bd2a26a76fe21f786e4fbd80607f,delivered,20180405,20180405.0,20180407.0,20180407.0,20180419,193316,194859.0,...,59.0,evening,0.0,38.0,52.0,night,16.0,12.0,43.0,afternoon
1,44e608f2db00c74a1fe329de44416a4e,00053a61a98854899e70ed204dd4bafe,delivered,20180228,20180228.0,20180301.0,20180316.0,20180326,111541,113242.0,...,42.0,morning,20.0,9.0,30.0,evening,19.0,2.0,51.0,evening
2,67503374d1fbcbe5e3a40324f703ffc8,00082cbe03e478190aadbea78542e933,delivered,20171119,20171121.0,20171122.0,20171129.0,20171207,152202,35224.0,...,24.0,night,12.0,3.0,26.0,afternoon,21.0,55.0,29.0,evening
3,34801c59d6ec5c5e89bfefb6ebef314d,000c8bdb58a29e7115cfc257230fb21b,delivered,20171212,20171212.0,20171215.0,20171227.0,20180105,225335,234229.0,...,29.0,evening,14.0,4.0,54.0,afternoon,1.0,37.0,12.0,night
4,c6d61340bd8baeedca7cc8e7f7ec07e9,000de6019bb59f34c099a907c151d855,delivered,20170817,20170817.0,20170818.0,20170821.0,20170906,191033,195522.0,...,22.0,evening,14.0,13.0,7.0,afternoon,19.0,18.0,42.0,evening


In [7]:
donnees["fact_order_item"].toPandas().head(5)

,order_id,order_item_id,purchase_timestamp,purchase_date_key,purchase_time_key,customer_unique_id,product_id,category_name,seller_id,price,...,purchase_date_month,purchase_date_month_name,purchase_date_iso_week,purchase_date_day,purchase_date_weekday,purchase_date_is_weekend,purchase_time_hour,purchase_time_minute,purchase_time_second,purchase_time_day_part
0,01b330808c5819a6a3cb79b72f0b8288,1,2018-03-12 15:22:12,20180312,152212,0005ef4cd20d2893f0d9fbd94d3c0d97,4fb3e6ce442ce9c621ea9509e47ac951,sports_leisure,3c03b12bab54d8b37d79d914bfdb1aa0,104.90,...,3,March,11,12,2,False,15,22,12,afternoon
1,85bf8863657bff31006811d45d1c8db9,1,2018-03-24 14:44:41,20180324,144441,00090324bbad0e9342388303bb71ba0a,f6fa93875f4a7dac03e0a22143f4d815,bed_bath_table,391fc6631aebcf3004804e51b40bcf1e,49.95,...,3,March,12,24,7,True,14,44,41,afternoon
2,c5d3fae065a8090bae39f4f34b14cd06,1,2018-06-25 09:53:57,20180625,95357,000e309254ab1fc5ba99dd469d36bdb4,b73091f98b819cf7a339f74ece86aa4b,fashion_underwear_beach,a6fe7de3d16f6149ffe280349a8535a0,59.90,...,6,June,26,25,2,False,9,53,57,morning
3,4b2626de72497770ddfa81a533f70c76,1,2018-05-20 16:26:06,20180520,162606,000ed48ceeb6f4bf8ad021a10a3c7b43,d2f5484cbffe4ca766301b21ab9246dd,computers_accessories,36a968b544695394e4e9d7572688598f,12.88,...,5,May,20,20,1,True,16,26,6,afternoon
4,3147d340c802cf04b463c4a5e532745d,1,2017-08-05 14:08:47,20170805,140847,0011c98589159d6149979563c504cb21,5c0826f946c32406d5baa79544668b4c,small_appliances,f9244d45189d3a3605499abddeade7d5,99.99,...,8,August,31,5,7,True,14,8,47,afternoon


In [8]:
donnees["fact_payment"].toPandas().head(5)

,order_id,payment_type,customer_unique_id,purchase_date_key,purchase_time_key,payment_value,installments,transaction_count,event_ts,customer_customer_id,...,purchase_date_month,purchase_date_month_name,purchase_date_iso_week,purchase_date_day,purchase_date_weekday,purchase_date_is_weekend,purchase_time_hour,purchase_time_minute,purchase_time_second,purchase_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,aggregated,0000f6ccb0745a6a4b88665a16c9f078,20171012,202941,43.62,0,1,2017-10-12T20:29:41Z,914991f0c02ef0843c0e7010c819d642,...,10,October,41,12,5,False,20,29,41,evening
1,d957021f1127559cd947b62533f484f7,aggregated,0004aac84e0df4da2b147fca70cf8255,20171114,194542,196.89,0,1,2017-11-14T19:45:42Z,47227568b10f5f58a524a75507e6992c,...,11,November,46,14,3,False,19,45,42,evening
2,d0028facea13f508e880202d7097a5a1,aggregated,00050ab1314c0e55a6ca13cf7181fecf,20180420,125723,35.38,0,1,2018-04-20T12:57:23Z,d2509c13692836fc0449e88cf9eb4858,...,4,April,16,20,6,False,12,57,23,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,aggregated,0005e1862207bf6ccc02e4228effd9a0,20170304,233212,150.12,0,1,2017-03-04T23:32:12Z,3b37fb626fdf46cd99d37ec62afa88ff,...,3,March,9,4,7,True,23,32,12,evening
4,6681163e3dab91c549952b2845b20281,aggregated,0006fdc98a402fceb4eb0ee528f6a8d4,20170718,92310,29.00,0,1,2017-07-18T09:23:10Z,2f29573c8cac5a7be11c5b649078f944,...,7,July,29,18,3,False,9,23,10,morning


In [9]:
donnees["fact_review"].toPandas().head(5)

,order_id,customer_unique_id,status,review_count,score_sum,average_score,text_review_groups,answer_hours_sum,creation_hours_sum,purchase_date_key,...,purchase_date_month,purchase_date_month_name,purchase_date_iso_week,purchase_date_day,purchase_date_weekday,purchase_date_is_weekend,purchase_time_hour,purchase_time_minute,purchase_time_second,purchase_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,delivered,1,4,4.0,1,24.0,483.0,20171012,...,10,October,41,12,5,False,20,29,41,evening
1,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,delivered,1,5,5.0,1,118.0,316.0,20171114,...,11,November,46,14,3,False,19,45,42,evening
2,d0028facea13f508e880202d7097a5a1,00050ab1314c0e55a6ca13cf7181fecf,delivered,1,4,4.0,1,92.0,179.0,20180420,...,4,April,16,20,6,False,12,57,23,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,0005e1862207bf6ccc02e4228effd9a0,delivered,1,4,4.0,1,147.0,120.0,20170304,...,3,March,9,4,7,True,23,32,12,evening
4,6681163e3dab91c549952b2845b20281,0006fdc98a402fceb4eb0ee528f6a8d4,delivered,1,3,3.0,1,119.0,398.0,20170718,...,7,July,29,18,3,False,9,23,10,morning


In [10]:
donnees["mart_customer_rfm"].toPandas().head(5)

,customer_unique_id,last_purchase,frequency,monetary,average_order_value,recency_days,last_purchase_date_key,last_purchase_time_key,event_ts,customer_customer_id,...,last_purchase_date_month,last_purchase_date_month_name,last_purchase_date_iso_week,last_purchase_date_day,last_purchase_date_weekday,last_purchase_date_is_weekend,last_purchase_time_hour,last_purchase_time_minute,last_purchase_time_second,last_purchase_time_day_part
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,1,141.90,141.90,3084,20180510,105627,2018-05-10T10:56:27Z,fadbb3709178fc513abc1b2670aa1ad2,...,5,May,19,10,5,False,10,56,27,morning
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,1,27.19,27.19,3087,20180507,111127,2018-05-07T11:11:27Z,4cb282e167ae9234755102258dd52ee8,...,5,May,19,7,2,False,11,11,27,morning
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,1,86.22,86.22,3510,20170310,210503,2017-03-10T21:05:03Z,9b3932a6253894a02c1df9d19004239f,...,3,March,10,10,6,False,21,5,3,evening
3,000bfa1d2f1a41876493be685390d6d3,2017-09-29 14:46:58,1,46.85,46.85,3307,20170929,144658,2017-09-29T14:46:58Z,e36917a47d5bf39bd3f38bcafb3d438f,...,9,September,39,29,6,False,14,46,58,afternoon
4,001147e649a7b1afd577e873841632dd,2017-08-31 17:12:56,1,212.16,212.16,3336,20170831,171256,2017-08-31T17:12:56Z,5247e5c7e9037e74343f13bbd8800a6a,...,8,August,35,31,5,False,17,12,56,afternoon
